# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I have chosen **Lane 2: Refresh / Content Opportunity Scoring**.

My provisional lane is about deciding which existing pages should be reviewed first when editorial time is limited. The unit of analysis is one pseudonymized content item/page. The output I want to build is not just a prediction; it is a ranked review queue with scores, reason codes, and a suggested action such as refresh, expand, protect, monitor, or prune.

This lane looks useful because content performance can decay for many reasons at once: demand changes, search position shifts, CTR weakens, engagement drops, and older pages may become stale. A single hand-written rule can catch some obvious cases, but the starter data suggests the opportunity set is much larger and messier than one freshness rule.

## 2. The question: decision, action, cost of a wrong call

- **Research question:** Which pages should a content team review first for refresh or monitoring, using safe observed search and engagement signals?
- **Decision improved:** Choosing the next pages to spend editorial review time on.
- **Who acts and what do they do?** A content editor, SEO specialist, or content strategist inspects the ranked queue, checks the reason codes, and decides whether to refresh, expand, protect, monitor, consolidate, or prune the page.
- **Output:** A top-K ranked queue of page candidates, with a score and plain-language reason codes. Precision@K is a natural metric because the real workflow is reviewing the top set first, not classifying every page perfectly.
- **Cost of a wrong call:** A false positive wastes review time on a page that may not need action. A false negative leaves a potentially valuable declining page unreviewed, so traffic loss may continue. The cost means the system should support human review rather than claim automatic decisions.
- **Why data or ML can help:** The decision depends on several interacting signals: impressions, clicks, position, CTR, engagement, content age, and freshness. Data can rank candidates more consistently than a single static rule, while the model still needs reason codes and validation before anyone acts on it.

## 3. Quick look at the data (2-3 real numbers)

The starter dataset supports Lane 2 as a practical review-queue problem:

1. The starter CSV has **30,000 pages across 32 pseudonymized clients**, so the task is large enough that manual triage would be expensive.
2. **16,262 pages (54.21%) are currently marked as down** by `trend_direction`, so observed decline is common in this slice. This is a starter proxy, not a future-outcome label.
3. Among pages with at least 500 impressions, **9,961 are down**, while a simple stale-and-visible rule (`days_since_last_update >= 180` and `impressions_90d >= 500`) finds only **17** pages. That gap is a good reason to study richer ranking signals instead of relying only on page age.

The code below recomputes those numbers from the starter CSV.

In [1]:
import pandas as pd

# Load the starter CSV from the notebook's folder: work/notebooks/ -> repo root.
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

total_pages = len(df)
total_clients = df["client_id"].nunique()
declining_pages = (df["trend_direction"] == "down").sum()
declining_pct = (df["trend_direction"] == "down").mean() * 100
visible_declining = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 500)).sum()
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()

print(f"Total content items/pages: {total_pages:,}")
print(f"Unique clients: {total_clients}")
print(f"Pages marked down by trend_direction: {declining_pages:,} ({declining_pct:.2f}%)")
print(f"Down pages with impressions_90d >= 500: {visible_declining:,}")
print(f"Stale visible pages by a simple rule: {stale_visible:,}")

Total content items/pages: 30,000
Unique clients: 32
Pages marked down by trend_direction: 16,262 (54.21%)
Down pages with impressions_90d >= 500: 9,961
Stale visible pages by a simple rule: 17


## 4. Careful words: what I can and can't claim

- **What I can claim now:** In the starter slice, many pages show an observed down trend, and the candidate set is not captured well by one simple freshness rule. This supports studying a ranked review queue for refresh opportunity.
- **What I cannot claim now:** I cannot claim that refreshing a page will cause recovery, that the model predicts future decline yet, or that these patterns prove anything about search engine algorithms.
- **Leakage guard:** `trend_direction` and `trend_pct` define the starter label/proxy, so they should not be used as ordinary model features. In later warehouse work, a stronger target should use a future window after the feature window.
- **How I will phrase results:** decision-support, observed, directional, and human-reviewed. The capstone should recommend which pages to inspect first, not promise guaranteed traffic gains.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.